# Test `frame_curation.py` on a SuperMemory-VQA video

Runs both curation methods (**uniform** and **CLIP + KMeans**) on one real session, plays the outputs, shows the chosen keyframes,
and checks how many object-question **answer-evidence spans** each method hits at the same frame budget.

**Before you run**
1. **Runtime → Change runtime type → T4 GPU.**
2. 🔑 **Secrets** (left sidebar): add `HF_TOKEN` and turn on *Notebook access*.
3. Upload **`frame_curation.py`** (a copy of `server/hindsight/video/curation.py`) with the 📁 Files sidebar (drag it into `/content`). If it's missing, the next cell asks for it.

The default video, `Person_10_session_3`, is about **1 GB** and has 12 object-question evidence spans. Total runtime is about 5–10 minutes.

In [ ]:
import os, sys
if not os.path.exists("/content/frame_curation.py"):
    from google.colab import files
    print("Upload frame_curation.py")
    files.upload()
sys.path.insert(0, "/content")
!pip -q install open_clip_torch
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# ---- settings ----
import os
VIDEO_ID = "Person_10_session_3_04112026_glasses_1264"   # ~1 GB, 12 evidence spans
# other options: "Person_8_session_3_04112026_glasses_1203" (0.9 GB, 8 spans),
#                "Person_5_session_1_03282026_glasses_1203" (2.1 GB, 34 spans)
K          = 32      # KMeans clusters = frames kept; uniform is matched to the same budget for the comparison
EMBED_FPS  = 1.0     # embed 1 frame/s (fast). None = every frame (~30x slower)
TOL_S      = 2.0     # a kept frame within +/- TOL_S seconds of an evidence span counts as a hit
REPO_ID    = "OSU-AIoT-MLSys-Lab/SuperMemory-VQA"
WORK       = "/content/fc_test"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# ---- Hugging Face login (Colab secret HF_TOKEN) and download ----
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
login(token=userdata.get("HF_TOKEN"))

person = VIDEO_ID.split("_session")[0]                       # e.g. "Person_10"
VIDEO = hf_hub_download(REPO_ID, f"data/video/{person}/{VIDEO_ID}.mp4", repo_type="dataset", local_dir=WORK)
QA_PATH = hf_hub_download(REPO_ID, "data/json/all_qa.json", repo_type="dataset")
print(VIDEO, f"{os.path.getsize(VIDEO)/1e9:.2f} GB")

## Look at the source video
Aria RGB frames are often **rotated**. If the frame below is sideways, set `io.rotate` (90, 180 or 270) in the next cell.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
try:   # repo checkout (server/ on sys.path)
    from hindsight.video.curation import FrameCurator, probe_video, iter_frames, fmt_ts
except ImportError:   # standalone copy uploaded to Colab as frame_curation.py
    from frame_curation import FrameCurator, probe_video, iter_frames, fmt_ts

info = probe_video(VIDEO)
print(f"{info.fps:.2f} fps, {info.n_frames} frames, {info.duration/60:.1f} min, {info.width}x{info.height}")
_, f0 = next(iter_frames(VIDEO, [int(info.n_frames // 2)]))
plt.figure(figsize=(4, 4)); plt.imshow(cv2.cvtColor(f0, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("middle frame"); plt.show()

In [ ]:
ROTATE = 0          # change to 90 / 180 / 270 if the frame above is sideways

cur = FrameCurator({
    "io": {"rotate": ROTATE, "resize": [448, 448], "overlay_timestamp": True,
           "reencode_h264": True},            # H.264 so the video plays inside Colab
    "uniform": {"fps": 1.0},
    "kmeans": {"k": K, "embed_fps": EMBED_FPS, "cache_dir": f"{WORK}/cache",
               "embedder": {"backend": "open_clip", "model": "ViT-B-32",
                            "pretrained": "laion2b_s34b_b79k", "device": "auto", "batch_size": 128}},
})

## Method 1: uniform sampling

In [ ]:
import time
t = time.time()
r_uni = cur.curate(VIDEO, f"{WORK}/uniform_1fps.mp4", method="uniform", fps=1.0)
print(f"uniform 1 fps: {len(r_uni.indices)} frames in {time.time()-t:.0f}s -> {r_uni.output}")

# same budget as KMeans (K frames spread over the whole video), used for the comparison below
fps_budget = K / info.duration
r_uniK = cur.curate(VIDEO, f"{WORK}/uniform_K.mp4", method="uniform", fps=fps_budget)
print(f"uniform at K budget: {len(r_uniK.indices)} frames ({fps_budget:.4f} fps)")

## Method 2: CLIP + KMeans

In [ ]:
t = time.time()
r_km = cur.curate(VIDEO, f"{WORK}/kmeans_K{K}.mp4", method="kmeans")
print(f"kmeans: {len(r_km.indices)} frames in {time.time()-t:.0f}s -> {r_km.output}")
print(r_km.extra)

In [ ]:
# play the outputs inside Colab
from IPython.display import HTML, display
from base64 import b64encode
def show(path, width=448):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<p>{os.path.basename(path)}</p><video width={width} controls src="data:video/mp4;base64,{data}"></video>'))
show(r_km.output)
show(r_uniK.output)

In [ ]:
# keyframe grids: KMeans vs uniform at the same budget
import json
def grid(result, title, cols=8):
    man = json.load(open(result.manifest))["frames"]
    frames = dict(iter_frames(VIDEO, result.indices))
    rows = int(np.ceil(len(man) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2.2 * rows))
    for ax in np.ravel(axes): ax.axis("off")
    for ax, m in zip(np.ravel(axes), man):
        img = cv2.cvtColor(frames[m["src_index"]], cv2.COLOR_BGR2RGB)
        if ROTATE: img = np.ascontiguousarray(np.rot90(img, k=-(ROTATE // 90)))
        ax.imshow(img)
        lab = m["t_str"] + (f"  n={m['cluster_size']}" if "cluster_size" in m else "")
        ax.set_title(lab, fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

grid(r_km, f"KMeans keyframes (K={K}); n = frames in that cluster")
grid(r_uniK, f"Uniform keyframes (same budget)")

## Do the kept frames hit the answer-evidence spans?
For every answerable **Object & Location Memory** question whose answer evidence is in this video, check whether a kept frame lands inside the evidence span.
If the question was asked in this same video, only frames before the question time count.

In [ ]:
import pandas as pd

def to_sec(x):
    if isinstance(x, (int, float)): return float(x)
    s = 0.0
    for p in str(x).split(":"): s = s * 60 + float(p)
    return s
def spans_of(ev):
    sp = ev.get("time_spans") or ([ev["time_span"]] if ev.get("time_span") else [])
    return [(to_sec(s["start_time"]), to_sec(s["end_time"])) for s in sp]

QA = json.load(open(QA_PATH))
rows = []
for r in QA:
    if r["metadata"]["skill"] != "object_location_memory" or not r["is_answerable"]:
        continue
    qs = spans_of(r["question_evidence"])
    q_cut = min(s for s, _ in qs) if (qs and r["metadata"]["primary_video_id"] == VIDEO_ID) else np.inf
    for e in r["answer_evidence"]["evidence_list"]:
        if e["video_id"] == VIDEO_ID:
            (s, en), = spans_of(e)
            rows.append(dict(question=r["question"], answer=r["correct_answer"], s=s, e=en, q_cut=q_cut))
EV = pd.DataFrame(rows)
print(f"{len(EV)} evidence spans in {VIDEO_ID}")

def hits(result):
    t = result.timestamps
    return [bool(((t >= s - TOL_S) & (t <= e + TOL_S) & (t <= q)).any()) for s, e, q in zip(EV.s, EV.e, EV.q_cut)]

rng = np.random.default_rng(0)
def random_hits(b, seeds=20):
    out = []
    for sd in range(seeds):
        t = np.sort(np.random.default_rng(sd).choice(info.n_frames, b, replace=False)) / info.fps
        out.append(np.mean([((t >= s - TOL_S) & (t <= e + TOL_S) & (t <= q)).any() for s, e, q in zip(EV.s, EV.e, EV.q_cut)]))
    return float(np.mean(out))

EV["uniform_K"] = hits(r_uniK)
EV["kmeans_K"]  = hits(r_km)
EV["uniform_1fps"] = hits(r_uni)
pd.set_option("display.max_colwidth", 80)
display(EV[["question", "s", "e", "uniform_K", "kmeans_K", "uniform_1fps"]])

summary = pd.DataFrame({
    "frames kept": [len(r_uniK.indices), len(r_km.indices), K, len(r_uni.indices)],
    "evidence spans hit (%)": [100 * EV.uniform_K.mean(), 100 * EV.kmeans_K.mean(), 100 * random_hits(K), 100 * EV.uniform_1fps.mean()],
}, index=["Uniform (K budget)", f"KMeans (K={K})", "Random (K budget, 20 seeds)", "Uniform 1 fps (upper bound)"])
display(summary.round(1))

## Try more settings
- Change `K` (for example 16, 64, 128) and rerun the KMeans cell. Embeddings are cached, so only KMeans reruns.
- `EMBED_FPS = None` embeds **every** frame (the original spec). It's slower but uses every frame.
- Swap the CLIP model, for example
  `cur.config["kmeans"]["embedder"] = {"backend": "hf", "model": "google/siglip2-base-patch16-224", "device": "auto", "batch_size": 64}`.
- One video and about 12 spans is a **smoke test**, not a result. Use the KMeans notebook for the multi-session evaluation.